# Knowledge Graph Reasoning

Owner: Member 4

This notebook checks whether linked claims are supported, contradicted, or unknown using conservative knowledge-graph reasoning rules.

## Role in the Project

Notebook 3 links extracted subjects and objects to Wikidata entities. This notebook uses those links to produce a reasoning result for every claim. The output `04_KG_Reasoning/kg_results.json` is the expected input for Bayesian inference in Notebook 5.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '04_KG_Reasoning'
sys.path.insert(0, str(MODULE_DIR))

from run_kg_reasoning import run_kg_reasoning

INPUT_PATH = PROJECT_ROOT / '03_Entity_Linking_KR' / 'linked_entities.json'
OUTPUT_PATH = MODULE_DIR / 'kg_results.json'
SUMMARY_PATH = MODULE_DIR / 'kg_reasoning_summary.json'

INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

## Input Preview

The input contains linked subject/object entities, optional Wikidata property IDs, and entity-linking confidence from Notebook 3.

In [ ]:
linked_records = json.loads(INPUT_PATH.read_text())
linked_records[:3]

## Reasoning Method

The rules are intentionally conservative:

- use property IDs only when comparison evidence is available
- support or contradict only narrow class claims using Wikidata descriptions
- treat speech relations such as `say` as `unknown`
- treat low-confidence entity links as `unknown`
- preserve every claim so Bayesian inference receives a complete handoff

In [ ]:
results = run_kg_reasoning(PROJECT_ROOT)
summary = results['summary']
summary

## Output Preview

Each output record contains the required KG fields: `claim_id`, `kg_status`, `kg_confidence`, `evidence`, `reasoning_rule`, and `source`. Extra context fields are kept for traceability.

In [ ]:
kg_df = results['results_df']
kg_df.head()

## Status Distribution

This table is useful for the empirical analysis section of the report.

In [ ]:
kg_df['kg_status'].value_counts()

## Supported and Contradicted Examples

Because the rules are conservative, most records will remain `unknown`. Any supported or contradicted rows should be inspected before they are used in the report.

In [ ]:
kg_df[kg_df['kg_status'].isin(['supported', 'contradicted'])][[
    'claim_id', 'kg_status', 'kg_confidence', 'subject', 'relation', 'object', 'evidence'
]]

## Limitations

The KG reasoning output is limited by the quality of the extracted triples and entity links. Many relations from Notebook 1 are generic verbs such as `say`, `be`, and `have`, so they cannot be safely converted into Wikidata claims. The system therefore favours `unknown` over unsupported support/contradiction decisions.